In [1]:
import os
# 1. [关键] 必须设置缓存到数据盘 (50G硬盘保命设置)
os.environ["HF_HOME"] = "/root/autodl-tmp/hf_cache"
# 2. [关键] 关闭 HF_TRANSFER 加速 (解决 RuntimeError: no permits available)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
# 3. [关键] 关闭 Xet 加速 (解决 CAS service error)
os.environ["HF_HUB_DISABLE_XET"] = "1"
# 4. [关键] 使用国内镜像 (解决连接超时)
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import sys
sys.path.append("..")
import torch
from unsloth import FastLanguageModel
from src.pipeline.qwen3_pipeline import Qwen3CoTPipeline
from src.utils.config_loader import load_config

cfg = load_config("../configs/config.yaml")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
MODEL_PATH = "../outputs/qwen3_cot_finetuned"
max_seq_length = cfg.model_student.max_seq_length
load_in_4bit = cfg.model_student.load_in_4bit
dtype = cfg.model_student.dtype
trust_remote_code = cfg.model_student.trust_remote_code

In [ ]:
print(f"⏳ Loading model from {MODEL_PATH}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    trust_remote_code = trust_remote_code, 
)

⏳ Loading model from ../outputs/qwen3_cot_finetuned...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth 2025.11.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [4]:
pipeline = Qwen3CoTPipeline(model, tokenizer)

In [ ]:
from datasets import load_dataset, concatenate_datasets
rest_dataset = load_dataset("json", data_files="../data/processed/trial_rest_clean.jsonl", split="train")
lap_dataset = load_dataset("json", data_files="../data/processed/trial_lap_clean.jsonl", split="train")

trial_dataset = concatenate_datasets([rest_dataset, lap_dataset])

print(f"✅ 成功加载测试集！总条数: {len(trial_dataset)}")

✅ 成功加载测试集！总条数: 200


In [13]:
import random
def pick_random_mixed(dataset):
    """
    从数据集中随机抽取一条，并智能识别是 Aspect Term 还是 Aspect Category
    """
    # 1. 随机选一条评论 (Text)
    sample_data = random.choice(dataset)
    text = sample_data['text']
    
    # 2. 建立候选池 (Candidate Pool)
    # 我们把所有可能的测试点都收集起来
    candidates = []
    
    # A. 收集具体 Aspect Terms (比如 "aluminum body", "steak")
    if 'aspectTerms' in sample_data and sample_data['aspectTerms']:
        for item in sample_data['aspectTerms']:
            candidates.append({
                "target": item['term'],       # 目标词
                "polarity": item['polarity'], # 真值
                "type": "Explicit Term"    # 类型标记
            })
            
    # B. 收集抽象 Aspect Categories (比如 "food", "price")
    # 这里响应你的需求，加入对 Category 的支持
    if 'aspectCategories' in sample_data and sample_data['aspectCategories']:
        for item in sample_data['aspectCategories']:
            candidates.append({
                "target": item['category'],   # 目标类别 (如 'food')
                "polarity": item['polarity'], # 真值
                "type": "Category"        # 类型标记
            })
    
    # 3. 如果这条数据既没 Term 也没 Category (脏数据)，重抽
    if not candidates:
        return pick_random_mixed(dataset)
    
    # 4. 从候选池中随机选一个作为题目
    selection = random.choice(candidates)
    
    # 返回：原文，目标(Term/Category)，真值，类型标记
    return text, selection['target'], selection['polarity'], selection['type']

In [17]:
text, target_aspect, ground_truth, selection_type = pick_random_mixed(trial_dataset)
pipeline.predict(text, target_aspect, stream=True)

<think>
1. Identify the target aspect: “anecdotes/miscellaneous.”  
2. Examine the sentence structure:  
   - The sentence is a conditional clause: “If only they delivered, they’d make a mint!”  
   - The clause “If only they delivered” expresses a wish or regret that the subject did not deliver.  
   - The clause “they’d make a mint” is a future hypothetical: if delivery occurred, the subject would achieve a significant gain (a “mint” is slang for a large amount of money).  
3. Determine the emotional tone:  
   - The phrase “If only” signals a negative sentiment toward the current state (they did not deliver).  
   - The hypothetical future outcome (“they’d make a mint”) is positive, but it is conditional on an unmet expectation.  
4. Assess the sentiment toward the target aspect:  
   - The target aspect is “anecdotes/miscellaneous.”  
   - The sentence does not explicitly describe a specific anecdote; it merely expresses a wish about a delivery.  
   - The overall tone is one of di

{'input_text': "If only they delivered, they'd make a mint!",
 'input_aspect': 'anecdotes/miscellaneous',
 'full_response': '<think>\n1. Identify the target aspect: “anecdotes/miscellaneous.”  \n2. Examine the sentence structure:  \n   - The sentence is a conditional clause: “If only they delivered, they’d make a mint!”  \n   - The clause “If only they delivered” expresses a wish or regret that the subject did not deliver.  \n   - The clause “they’d make a mint” is a future hypothetical: if delivery occurred, the subject would achieve a significant gain (a “mint” is slang for a large amount of money).  \n3. Determine the emotional tone:  \n   - The phrase “If only” signals a negative sentiment toward the current state (they did not deliver).  \n   - The hypothetical future outcome (“they’d make a mint”) is positive, but it is conditional on an unmet expectation.  \n4. Assess the sentiment toward the target aspect:  \n   - The target aspect is “anecdotes/miscellaneous.”  \n   - The sent